In [65]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cup")
print(device)

cuda


In [66]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [67]:
import datasets
print(datasets.__file__)
from datasets import load_dataset

# dataset = load_dataset("")

/usr/local/lib/python3.12/dist-packages/datasets/__init__.py


In [68]:
from datasets import load_dataset
import inspect

print(inspect.getfile(load_dataset))   # file path
#print(inspect.getsource(load_dataset)) # actual source code
dataset = load_dataset("bentrevett/multi30k")
sample = iter(dataset["train"])
train = dataset["train"]
val = dataset["validation"]
test = dataset["test"]


/usr/local/lib/python3.12/dist-packages/datasets/load.py


In [69]:
from datasets import load_dataset

dataset = load_dataset("bentrevett/multi30k")
print(dataset)
sample = iter(dataset["train"])
train = dataset["train"]
val = dataset["validation"]
test = dataset["test"]
print(sample)


DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})
<generator object Dataset.__iter__ at 0x7bbe027e1cc0>


In [70]:
# !pip uninstall -y torch torchtext
# !pip install torch==2.3.1 torchtext==0.18.0

In [71]:
token_transform = {}
vocab_transform = {}

from torchtext.data.utils import get_tokenizer


In [72]:
# !python -m spacy download de_core_news_sm

In [127]:
token_transform['en'] = get_tokenizer('spacy', language='en_core_web_sm')
token_transform['de'] = get_tokenizer('spacy', language='de_core_news_sm')


In [128]:
print("sentence:", next(sample))
print("Tokenization:", token_transform['en'](next(sample)['en']))


sentence: {'en': 'A man in a blue shirt is standing on a ladder cleaning a window.', 'de': 'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.'}
Tokenization: ['Two', 'men', 'are', 'at', 'the', 'stove', 'preparing', 'food', '.']


In [129]:
#a function to tokenize the inputs

def yield_tokens(data, language):
    language_index = {'en':0, 'de':1}

    for data_sample in data:
        yield token_transform[language](data_sample[language])

In [130]:
print(next(yield_tokens(sample, 'de')))

['Ein', 'Mann', 'in', 'grün', 'hält', 'eine', 'Gitarre', ',', 'während', 'der', 'andere', 'Mann', 'sein', 'Hemd', 'ansieht', '.']


In [131]:
#define special symbols & indices

UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0,1,2,3

# Make sure the tokens are in the order of their indicies to properly insert them in vocab

special_symbols = ['<unk>', "<pad>", "<sos>", "<eos>"]

In [132]:
# text to integers (Numericalization)

from torchtext.vocab import build_vocab_from_iterator
#build_vocab_from_iterator?
for ln in ['en', 'de']:
    vocab_transform[ln] = build_vocab_from_iterator(
        yield_tokens(dataset["train"], ln),  # <-- use dataset directly
        min_freq=2,
        specials=special_symbols,
        special_first=True
    )

for ln in ['en', 'de']:
    vocab_transform[ln].set_default_index(UNK_IDX)

vocab_transform['en'](['here', 'is', 'a', 'unknowwww', 'a'])


[2207, 10, 4, 0, 4]

In [133]:
mapping = vocab_transform['en'].get_itos()
mapping[-1]
len(mapping)
mapping[1891]

'case'

In [134]:
# Assuming you have standard BOS (Beginning of Sequence) and EOS (End of Sequence) indices
# BOS_IDX = vocab_transform['en']['<bos>'] # Update with your actual BOS token if different
# EOS_IDX = vocab_transform['en']['<eos>'] # Update with your actual EOS token if different

SOS_IDX = vocab_transform['en']['<sos>']
EOS_IDX = vocab_transform['en']['<eos>']

def pre_process_dataset(raw_dataset):
    processed_data = []
    for item in raw_dataset:
        # 1. Tokenize the strings using SpaCy
        src_tokens = token_transform['en'](item['en'].rstrip("\n"))
        tgt_tokens = token_transform['de'](item['de'].rstrip("\n"))

        # 2. Convert string tokens to Vocabulary Integer IDs
        src_indices = vocab_transform['en'](src_tokens)
        tgt_indices = vocab_transform['de'](tgt_tokens)

        # 3. Add BOS and EOS tokens, and convert to PyTorch Tensors
        src_tensor = torch.tensor([SOS_IDX] + src_indices + [EOS_IDX], dtype=torch.long)
        tgt_tensor = torch.tensor([SOS_IDX] + tgt_indices + [EOS_IDX], dtype=torch.long)

        # Store the clean, ready-to-use tensors
        processed_data.append({
            'src': src_tensor,
            'tgt': tgt_tensor
        })
    return processed_data

print("Pre-processing training data...")
train_processed = pre_process_dataset(train)

print("Pre-processing validation data...")
valid_processed = pre_process_dataset(val)

print("Pre-processing testing data...")
test_processed = pre_process_dataset(test)

Pre-processing training data...


TypeError: 'function' object is not iterable

In [135]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

BATCH_SIZE = 64


#helper fn to club sequential operations togather
def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input   # ✅ THIS WAS MISSING
    return func

# function to add BOS/EOS & create tensor input sequence indices
def tensor_transform(token_ids):
    return torch.cat([
        torch.tensor([SOS_IDX], dtype=torch.long),
        torch.tensor(token_ids, dtype=torch.long),
        torch.tensor([EOS_IDX], dtype=torch.long)
    ])

#src & tgt language text transforms to convert raw strings into tensor indices

text_transform = {}
for ln in ['en', 'de']:
    text_transform[ln] = sequential_transforms(token_transform[ln],
                                               vocab_transform[ln],
                                               tensor_transform
                                               )

#fn to collate data samples into batche tensors

def collate_batch(batch):
    src_batch, tgt_batch, src_len_batch = [], [], []

    for item in batch:
        src_sample = item['en']
        tgt_sample = item['de']

        src_tensor = text_transform['en'](src_sample.rstrip("\n"))
        tgt_tensor = text_transform['de'](tgt_sample.rstrip("\n"))

        src_batch.append(src_tensor)
        tgt_batch.append(tgt_tensor)
        src_len_batch.append(src_tensor.size(0))

    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)

    return src_batch, torch.tensor(src_len_batch, dtype=torch.long), tgt_batch

In [136]:
# create train, val & test dataloaders

batch_size = 512

train_loader = DataLoader(train, batch_size=512, shuffle=True, collate_fn=collate_batch, num_workers=2, pin_memory=True)
valid_loader = DataLoader(val, batch_size=512, shuffle=True, collate_fn=collate_batch, num_workers=2, pin_memory=True)
test_loader = DataLoader(test, batch_size=512, shuffle=True, collate_fn=collate_batch, num_workers=2, pin_memory=True)

TypeError: object of type 'function' has no len()

In [137]:
# creating vocab
from torchtext.vocab import build_vocab_from_iterator

src_tokenizer = get_tokenizer("spacy", language='en_core_web_sm')
tgt_tokenizer = get_tokenizer("spacy", language='de_core_news_sm')

#iterators for tokenization
#I need to make convert this "vocab = get_tokenizer("spacy", language='en_core_web_sm')(item['ln'].rstrip("\n"))" into an iterator because build_vocab_from_iterator needs like this.
def src_yield_tokens(dataset):
    for data in dataset:
        yield src_tokenizer(data['en'].rstrip("\n"))

def tgt_yield_tokens(dataset):
    for data in dataset:
        yield tgt_tokenizer(data['de'].rstrip("\n"))

#now we got vocab_src and vocab_tgt objects
vocab_src = build_vocab_from_iterator(src_yield_tokens(train), min_freq=2, specials = ['<unk>', "<pad>", "<sos>", "<eos>"], special_first=True)
vocab_tgt = build_vocab_from_iterator(tgt_yield_tokens(train), min_freq=2, specials=['<unk>', "<pad>", "<sos>", "<eos>"], special_first=True)

vocab_src.set_default_index(UNK_IDX)
vocab_tgt.set_default_index(UNK_IDX)
#demo showing how to use these objects ->

print(vocab_src(["hi", "hello", "play", "cry"]))
print(len(vocab_src))

TypeError: 'function' object is not iterable

In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Update with your actual PAD index variable
PAD_IDX = vocab_transform['en']['<pad>']

def fast_collate_batch(batch):
    src_batch, tgt_batch, src_len_batch = [], [], []

    for item in batch:
        src_batch.append(item['src'])
        tgt_batch.append(item['tgt'])
        src_len_batch.append(len(item['src']))


    # Pad the sequences so they all match the longest sentence in THIS batch
    # Note: If your transformer expects shape [seq_len, batch_size], set batch_first=False
    src_padded = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_padded = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)

    return src_padded, torch.tensor(src_len_batch), tgt_padded

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128 # Much higher now that the CPU bottleneck is gone!

train_loader = DataLoader(
    train_processed,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=fast_collate_batch,
    num_workers=2,       # Use 2 background CPU cores to fetch data
    pin_memory=True      # Speeds up transfer from CPU RAM to GPU VRAM
)

valid_loader = DataLoader(
    valid_processed,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=fast_collate_batch,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_processed,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=fast_collate_batch,
    num_workers=2,
    pin_memory=True
)

In [ ]:
print(train_loader)

for en, en_len, de in train_loader:
    print(en)
    print(de)
    print(en_len)
    break

#print("English shape:" en.shape)
#print("German shape:" de.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)*(-math.log(10000)/d_model)
        )

        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        out = x + self.pe[:, :x.shape[1], :]

        return out


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        assert d_model%n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model//n_heads

        self.linear_q = nn.Linear(d_model, d_model)
        self.linear_k = nn.Linear(d_model, d_model)
        self.linear_v = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):

        bs, q_seq_len, _ = q.shape
        bs, k_seq_len, _ = k.shape
        bs, v_seq_len, _ = v.shape
        #bs = q.shape[0]

        q_proj = self.linear_q(q) # (bs, seq_len, d_model)
        k_proj = self.linear_k(k)
        v_proj = self.linear_v(v)

        q = q_proj.view(bs, q_seq_len, self.n_heads, self.head_dim).transpose(1,2) #(bs, n_heads, q_seq_len, head_dim)
        k = k_proj.view(bs, k_seq_len, self.n_heads, self.head_dim).transpose(1,2) #(bs, n_heads, k_seq_len, head_dim)
        v = v_proj.view(bs, v_seq_len, self.n_heads, self.head_dim).transpose(1,2) #(bs, n_heads, v_seq_len, head_dim)

        temp = torch.matmul(q, k.transpose(-1,-2))/math.sqrt(self.head_dim) #(bs, n_heads, q_seq_len, k_seq_len)

        if mask is not None:
            temp = temp.masked_fill(mask == 0, -1e9)

        attentions = torch.softmax(temp, dim = -1) #(bs, n_heads, q_seq_len, k_seq_len)

        #(bs, n_heads, q_seq_len, k_seq_len) * (bs, n_heads, v_seq_len, head_dim)
        out = torch.matmul(attentions, v) #(bs, n_heads, q_seq_len, head_dim)

        out = out.transpose(1,2).contiguous() #(bs, q_seq_len, n_heads, head_dim)
        out = out.view(bs, q_seq_len, self.d_model) #(bs, q_seq_len, d_model)

        out = self.fc_out(out)

        return out #(bs, q_seq_len, d_model)




In [138]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hid_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, hid_dim),
            nn.ReLU(),
            nn.Linear(hid_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [139]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, hid_dim, dropout):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, n_head)
        self.addNorm1 = nn.LayerNorm(d_model)
        self.ff1 = FeedForward(d_model, hid_dim)
        self.addNorm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x, src_mask):

        attn = self.self_attention(x, x, x, src_mask)
        x = self.addNorm1(x + self.dropout(attn)) #attn = (bs, seq_len, d_model), x = [bs, seq_len, d_model]

        ff_out = self.ff1(x)
        out = self.addNorm2(x + self.dropout(ff_out))

        return out


In [153]:
class Encoder(nn.Module):
    def __init__(self, d_model, n_head, hid_dim, src_vocab_size, max_len, dropout, num_layers):
        super().__init__()
        self.d_model = d_model
        self.embeddings = nn.Embedding(src_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(max_len, d_model)

        self.layers = nn.ModuleList(
            EncoderLayer(d_model, n_head, hid_dim, dropout) for _ in range(num_layers)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        x = self.embeddings(src)*math.sqrt(self.d_model)

        x = self.dropout(self.pos_enc(x))

        for layer in self.layers:
            x = layer(x, src_mask)

        return x

In [154]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, hid_dim, dropout):
        super().__init__()

        self.masked_self_attention = MultiHeadAttention(d_model, n_head)
        self.addNorm1 = nn.LayerNorm(d_model)

        self.cross_attention = MultiHeadAttention(d_model, n_head)
        self.addNorm2 = nn.LayerNorm(d_model)

        self.ff1 = FeedForward(d_model, hid_dim)
        self.addNorm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, enc_out, tgt_mask, src_mask):

        msk_attn = self.masked_self_attention(tgt, tgt, tgt, tgt_mask)
        tgt = self.addNorm1(tgt + self.dropout(msk_attn))

        crs_attn = self.cross_attention(tgt, enc_out, enc_out, src_mask)
        tgt = self.addNorm2(tgt + self.dropout(crs_attn))

        ff_out = self.ff1(tgt)
        out = self.addNorm3(tgt + self.dropout(ff_out))

        return out



In [155]:
class Decoder(nn.Module):
    def __init__(self, d_model, n_head, hid_dim, tgt_vocab_size, max_len, dropout, num_layers):
        super().__init__()
        self.d_model = d_model
        self.embeddings = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(max_len, d_model)

        self.layers = nn.ModuleList(
            DecoderLayer(d_model, n_head, hid_dim, dropout) for _ in range(num_layers)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, enc_out, tgt_mask, src_mask):
        x = self.embeddings(tgt)*math.sqrt(self.d_model)

        x = self.dropout(self.pos_enc(x))

        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, src_mask)

        return x

In [156]:
class Transformer(nn.Module):
    def __init__(self, d_model, n_head, hid_dim, tgt_vocab_size, max_len, dropout, num_layers, src_vocab_size):
        super().__init__()

        self.encoder = Encoder(d_model, n_head, hid_dim, src_vocab_size, max_len, dropout, num_layers)
        self.decoder = Decoder(d_model, n_head, hid_dim, tgt_vocab_size, max_len, dropout, num_layers)

        self.linear = nn.Linear(d_model, tgt_vocab_size)

    def src_mask_maker(self, src, SRC_PAD_IDX):
        # scr -> (bs, seq_len)
        # make a boolean mask
        src_mask = (src != SRC_PAD_IDX) # (bs, seq_len)
        src_mask = src_mask.unsqueeze(1).unsqueeze(2)

        return src_mask

    def tgt_mask_maker(self, tgt, TGT_PAD_IDX):

        bs, tgt_len = tgt.shape

        # padding mask
        tgt_pad_mask = (tgt != TGT_PAD_IDX)                  # (bs, tgt_len)
        tgt_pad_mask = tgt_pad_mask.unsqueeze(1).unsqueeze(2)  # (bs,1,1,tgt_len)

        # causal mask (future masking)
        tgt_sub_mask = torch.tril(
            torch.ones((tgt_len, tgt_len), device=tgt.device)
        ).bool()                                             # (tgt_len, tgt_len)

        tgt_sub_mask = tgt_sub_mask.unsqueeze(0).unsqueeze(1)  # (1,1,tgt_len,tgt_len)

        # combine both
        tgt_mask = tgt_pad_mask & tgt_sub_mask               # (bs,1,tgt_len,tgt_len)

        return tgt_mask

    def forward(self, src, tgt, SRC_PAD_IDX, TGT_PAD_IDX):

        src_mask = self.src_mask_maker(src, SRC_PAD_IDX)
        tgt_mask = self.tgt_mask_maker(tgt, TGT_PAD_IDX)

        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, tgt_mask, src_mask)

        out = self.linear(dec_out)


        return out


In [157]:
def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)

In [158]:
src_vocab_size = len(vocab_src)
tgt_vocab_size = len(vocab_tgt)
d_model = 256
hid_dim = 512
dropout = 0.3
SRC_PAD_IDX = PAD_IDX
TGT_PAD_IDX = PAD_IDX
n_head = 8
max_len = 1000
num_layers = 6

model = Transformer(d_model, n_head, hid_dim, tgt_vocab_size, max_len, dropout, num_layers, src_vocab_size)
model = model.to(device)
model.apply(initialize_weights)


Transformer(
  (encoder): Encoder(
    (embeddings): Embedding(6191, 256)
    (pos_enc): PositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attention): MultiHeadAttention(
          (linear_q): Linear(in_features=256, out_features=256, bias=True)
          (linear_k): Linear(in_features=256, out_features=256, bias=True)
          (linear_v): Linear(in_features=256, out_features=256, bias=True)
          (fc_out): Linear(in_features=256, out_features=256, bias=True)
        )
        (addNorm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ff1): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=256, out_features=512, bias=True)
            (1): ReLU()
            (2): Linear(in_features=512, out_features=256, bias=True)
          )
        )
        (addNorm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
    )
    (dropout): Dropout(p=

In [159]:
def count_parameters(model):
    params = [p.numel() for p in model.parameters() if p.requires_grad]
    for item in params:
        print(f'{item:>6}')
    print(f'_____\n{sum(params):>6}')

count_parameters(model)


1584896
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
2051584
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
131072
   512
131072
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   256
   256
 65536
   256
 65536
   256
 65536
   256
 65536
   256
   2

In [160]:
import torch.optim as optim

lr = 0.0001

#training hyperparameters
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss(ignore_index = PAD_IDX)

In [161]:
from tqdm import tqdm # Import this at the top of your cell

def train(model, loader, optimizer, criterion, clip, loader_length, SRC_PAD_IDX, TGT_PAD_IDX):
    model.train()
    epoch_loss = 0

    # Wrap your loader in tqdm to generate the progress bar
    # desc="Training" gives the bar a title
    pbar = tqdm(loader, total=loader_length, desc="Training", leave=False)

    for src, src_len, tgt in pbar:

        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()

        outputs = model(src, tgt[:, :-1], SRC_PAD_IDX, TGT_PAD_IDX)

        tgt_vocab_size = outputs.shape[-1]

        # loss functions only works on 2D inputs with 1D targets
        outputs = outputs.contiguous().view(-1, tgt_vocab_size)
        tgt = tgt[:, 1:].contiguous().view(-1)

        loss = criterion(outputs, tgt)
        loss.backward()

        # CLIP THE GRADIENTS TO PREVENT THEM FROM EXPLODING
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

        # UPDATE THE PROGRESS BAR:
        # This displays the current loss next to the loading bar so you can see it dropping!
        pbar.set_postfix({'batch_loss': f"{loss.item():.4f}"})

    return epoch_loss / loader_length

In [162]:
def evaluate(model, loader, criterion, clip, loader_length, SRC_PAD_IDX, TGT_PAD_IDX):

    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, src_len, tgt in loader:

            #src = src.to(device)
            #tgt = tgt.to(device)

            src = src.to(device)
            tgt = tgt.to(device)

            outputs = model(src, tgt[:, :-1], SRC_PAD_IDX, TGT_PAD_IDX)

            #tgt = [batch_size, seq_len]
            #output = [batch_size, seq_len, tgt_vocab_size]

            tgt_vocab_size = outputs.shape[-1]

            # loss functions only works on 2D inputs with 1D targets

            outputs = outputs.contiguous().view(-1, tgt_vocab_size)
            tgt = tgt[:, 1:].contiguous().view(-1)

            loss = criterion(outputs, tgt)

            #CLIP THE GRADIENTS TO PREVENT THEM FROM WXPLODING
            #torch.nn.utils.clip_grad_norm(model.parameters(), clip)


            epoch_loss += loss.item()

    return epoch_loss/loader_length


In [163]:
#actual training

train_loader_length = len(list(iter(train_loader)))
val_loader_length = len(list(iter(valid_loader)))
test_loader_length = len(list(iter(test_loader)))

In [164]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time/60)
    elapsed_secs = int(elapsed_time - (elapsed_mins*60))
    return elapsed_mins, elapsed_secs

In [166]:
import time
best_valid_loss = float('inf')
num_epoch = 20
clip = 1

save_path = f'{model.__class__.__name__}.pt'
train_losses, val_losses = [], []

for epoch in range(num_epoch):
    start_time = time.time()

    train_loss = train(model, train_loader, optimizer, criterion, clip, train_loader_length, SRC_PAD_IDX, TGT_PAD_IDX)
    valid_loss = evaluate(model, valid_loader, criterion, clip, val_loader_length, SRC_PAD_IDX, TGT_PAD_IDX)

    train_losses.append(train_loss)
    val_losses.append(valid_loss)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), save_path)

    print(f'Epoch: {epoch+1} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss): 7.3f}')
    print(f'\tValid Loss: {valid_loss:.3f}| Val.PPL: {math.exp(valid_loss):7.3f}')



Epoch: 1 | Time: 0m 26s
	Train Loss: 4.003 | Train PPL:  54.747
	Valid Loss: 3.857| Val.PPL:  47.340


Epoch: 2 | Time: 0m 28s
	Train Loss: 3.897 | Train PPL:  49.243
	Valid Loss: 3.753| Val.PPL:  42.667


Epoch: 3 | Time: 0m 28s
	Train Loss: 3.809 | Train PPL:  45.102
	Valid Loss: 3.686| Val.PPL:  39.875


Epoch: 4 | Time: 0m 28s
	Train Loss: 3.732 | Train PPL:  41.781
	Valid Loss: 3.595| Val.PPL:  36.408


Epoch: 5 | Time: 0m 28s
	Train Loss: 3.661 | Train PPL:  38.920
	Valid Loss: 3.540| Val.PPL:  34.460


...

Epoch: 16 | Time: 0m 28s
	Train Loss: 3.105 | Train PPL:  22.309
	Valid Loss: 3.025| Val.PPL:  20.604


Epoch: 17 | Time: 0m 28s
	Train Loss: 3.066 | Train PPL:  21.457
	Valid Loss: 2.980| Val.PPL:  19.678


Epoch: 18 | Time: 0m 28s
	Train Loss: 3.030 | Train PPL:  20.698
	Valid Loss: 2.949| Val.PPL:  19.094


Epoch: 19 | Time: 0m 28s
	Train Loss: 2.996 | Train PPL:  19.999
	Valid Loss: 2.928| Val.PPL:  18.681


Epoch: 20 | Time: 0m 28s
	Train Loss: 2.963 | Train PPL:  19.364
	Valid Loss: 2.898| Val.PPL:  18.137


In [167]:
def translate_sentence(sentence, model, device, max_len=50):
    # 1. Put the model in evaluation mode (turns off dropout!)
    model.eval()

    # Ensure we have our special token IDs ready
    SOS_IDX = vocab_transform['en']['<sos>']
    EOS_IDX = vocab_transform['en']['<eos>']

    # 2. Tokenize the input English string using SpaCy
    if isinstance(sentence, str):
        tokens = token_transform['en'](sentence)
    else:
        tokens = sentence

    # 3. Convert string tokens to Integer IDs and wrap them in <sos> and <eos>
    src_indexes = [SOS_IDX] + vocab_transform['en'](tokens) + [EOS_IDX]

    # 4. Convert to a tensor, add a batch dimension of 1, and send to GPU
    # Shape becomes: [1, source_sequence_length]
    src_tensor = torch.tensor(src_indexes, dtype=torch.long).unsqueeze(0).to(device)

    # 5. Initialize the target sequence with ONLY the <sos> token
    tgt_indexes = [vocab_transform['de']['<sos>']]

    # 6. The Autoregressive Loop
    for i in range(max_len):

        # Convert our growing list of target predictions into a tensor
        # Shape: [1, current_target_length]
        tgt_tensor = torch.tensor(tgt_indexes, dtype=torch.long).unsqueeze(0).to(device)

        # Turn off gradient tracking to save memory and speed up computation
        with torch.no_grad():
            # Pass both the English tensor and the current German tensor into the model
            output = model(src_tensor, tgt_tensor, SRC_PAD_IDX, TGT_PAD_IDX)

        # The output shape is [batch_size, seq_len, vocab_size].
        # We only care about the model's prediction for the VERY LAST token in the sequence.
        # .argmax(2) finds the vocabulary ID with the highest probability.
        pred_token = output.argmax(2)[:, -1].item()

        # Add the newly predicted token to our list
        tgt_indexes.append(pred_token)

        # If the model predicts the sentence is over, break the loop!
        if pred_token == vocab_transform['de']['<eos>']:
            break

    # 7. Convert the final list of Integer IDs back into German strings
    tgt_tokens = vocab_transform['de'].lookup_tokens(tgt_indexes)

    # 8. Return the translated words (slicing off the <sos> and <eos> tokens for a clean string)
    return tgt_tokens[1:-1]

In [168]:
# Type your sentence here
english_sentence = "A man in a blue shirt is walking down the street."

# Run the translation
translated_words = translate_sentence(english_sentence, model, device)

# Join the list of words into a single readable string
german_translation = " ".join(translated_words)

print(f"English: {english_sentence}")
print(f"German:  {german_translation}")

English: A man in a blue shirt is walking down the street.
German:  Ein Mann in einem blauen Hemd geht auf die Straße entlang .


In [ ]:
# Ein Mann in einem blauen Hemd geht die Straße entlang